In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import math
import statistics

import warnings
warnings.filterwarnings('ignore')

#sys.path.append('../common')
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold,StratifiedShuffleSplit
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import make_scorer,recall_score


from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# #-----
#from xgboost.sklearn import XGBClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


# import pprint

# from scipy import stats
import scipy.io as spio
import numpy as np
import pandas as pd
# import pprint
# import scipy.io as spio
# from tqdm import tqdm
# from timeit import default_timer as timer
import pickle



# pp = pprint.PrettyPrinter(indent=4, depth=3)

# plt.style.use('ggplot')
import category_encoders as ce
import lightgbm
import logging
import numpy as np
import os
import pandas as pd
#import pytest
import sklearn
import xgboost

import ads

from ads.hpo.stopping_criterion import *
from ads.hpo.distributions import *
from ads.hpo.search_cv import ADSTuner

from lightgbm import LGBMClassifier
from sklearn import preprocessing
from sklearn.compose import ColumnTransformer
#from sklearn.datasets import load_iris, load_boston
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostRegressor, AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier
from ads.model import GenericModel
from ads.model import DataScienceModel
import automlx
from automlx import init


In [ ]:
# outFile = open('demo.txt', 'w')
# pd.set_option('display.max_rows', 999)
# pd.set_option('display.max_columns', 999)
# pd.set_option('display.width', 999)
# np.set_printoptions(threshold=sys.maxsize)
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

<H3> Data </H3>

In [ ]:

blinks = pd.read_csv(os.path.join(os.getcwd(),'data/DLC_LOMO_dataset.csv'), sep=',', header=0)

blinks_df = pd.DataFrame(blinks)


In [ ]:
blinks_df

In [ ]:
from catboost import CatBoostClassifier

catboost_params = {
    'iterations': 235,
    'learning_rate': 0.7871746344459382,
    'leaf_estimation_method': 'Newton',
    'colsample_bylevel': 0.096865,
    'depth': 3,
    'l2_leaf_reg': 2.567326,
    'feature_border_type': 'UniformAndQuantiles',
    'model_size_reg': 3.85132,
    'leaf_estimation_iterations': 1,
    'boosting_type': 'Plain',
    'bootstrap_type': 'MVS',
    'auto_class_weights': 'SqrtBalanced',
    'allow_writing_files': False,
    'allow_const_label': True,
    'thread_count': 1,
    'verbose': False,
    'random_seed': 42
}

In [ ]:
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    precision_score,
    roc_auc_score
)

import numpy as np
import pandas as pd

# Preserve labels and group identifiers
label_df = blinks_df['scores'].copy()
groups = blinks_df['subject_id'].copy()

# Select the same six features used in the original AutoMLx classifier
cols_to_keep = [
    'min_p4',
    'max_euclid_dist',
    'max_ear',
    'max_eye_area',
    'max_p1',
    'max_p4'
]

# Create an independent DataFrame containing only these features
mean_df = blinks_df[cols_to_keep].copy()

# Initialize leave-one-subject-out validation
logo = LeaveOneGroupOut()

# Store results across all folds
results=[]

mean_df

In [ ]:
print(mean_df.columns.tolist())
print("Number of features:", mean_df.shape[1])

In [ ]:
for fold, (train_idx, test_idx) in enumerate(
    logo.split(mean_df, label_df, groups), start=1
):
    # Identify the excluded subject
    test_subject = groups.iloc[test_idx].iloc[0]

    # Select training and testing trials
    X_train = mean_df.iloc[train_idx].copy()
    X_test = mean_df.iloc[test_idx].copy()

    y_train = label_df.iloc[train_idx].copy()
    y_test = label_df.iloc[test_idx].copy()

    print(f'\n========FOLD {fold}========')
    print('Training subjects: ', groups.iloc[train_idx].unique())
    print('Testing subject:', test_subject)
    print('Training trials:',X_train.shape[0])
    print('Testing trials:',X_test.shape[0])

    # Initialize a new scaler for this fold
    min_max_scaler = preprocessing.MinMaxScaler()

    # Fit scaler using training subjects only
    X_train = pd.DataFrame(
        min_max_scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    # Apply same scaling to held-out subject
    X_test = pd.DataFrame(
        min_max_scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )

    # Initialize fresh CatBoost classifier
    model = CatBoostClassifier(**catboost_params)

    # Train using four subjects
    model.fit(X_train, np.asarray(y_train).ravel())

    # Predict blink labels for held-out subject
    y_pred = np.asarray(model.predict(X_test)).ravel()

    # Probability assigned to class 1
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Calculate classifier performance
    score_accuracy = accuracy_score(y_test,y_pred)

    score_balanced = balanced_accuracy_score(
        y_test,y_pred
    )

    score_recall = recall_score(
        y_test,y_pred,zero_division = 0
    )

    score_precision = precision_score(
        y_test,y_pred,zero_division=0
    )

    # ROC AUC requires both classes
    if y_test.nunique() == 2:
        score_roc_auc = roc_auc_score(
            y_test, y_pred_proba
    )
    else:
        score_roc_auc = np.nan

    # Save the fold's results
    results.append({
        'Fold': fold,
        'subject': test_subject,
        'N_train': len(train_idx),
        'N_test': len(test_idx),
        'Accuracy': score_accuracy,
        'Balanced Accuracy': score_balanced,
        'ROC AUC': score_roc_auc,
        'Recall': score_recall,
        'Precision': score_precision
    })

print(f'Accuracy: {score_accuracy:.4f}')
print(f'Balanced Accuracy: {score_balanced:.4f}')
print(f'ROC AUC: {score_roc_auc:.4f}')
print(f'Recall: {score_recall:.4f}')
print(f'Precision: {score_precision:.4f}')




In [ ]:
# Collect all subject-specific results
results_df = pd.DataFrame(results)

print('\nLOMO Results:')
print(results_df)

# Save subject-specific results
results_df.to_csv(
    'LOMO_CatBoost_FixedParams_results.csv',
    index = False
)

# Summarize performance across subjects
metrics = [
    'Accuracy',
    'Balanced Accuracy',
    'ROC AUC',
    'Recall',
    'Precision'
]

summary_df = pd.DataFrame({
    'Mean': results_df[metrics].mean(),
    'SEM': results_df[metrics].sem()
})

print('\n LOMO Summary:')
print(summary_df)

summary_df.to_csv('LOMO_CatBoost_FixedParams_summary.csv')